# MemWeave-Final: a leak-controlled, held-out evaluation of budget-constrained memory retention

**What this notebook is.** The last stage of the MemWeave project (v2.0 → v2.2 → v2.3 addendum). The earlier versions reported a large learned-selector gain, then your own audit showed most of it came from benchmark artifacts. This notebook rebuilds the evaluation so that every claim it prints is one the data can actually support.

| Weakness in v2.2 / v2.3 | What this notebook does about it |
|---|---|
| Learned edge over `low_novelty` was ~+0.03 and fragile | Primary hypothesis, comparison family, equivalence margin and the "best simple rule" are all fixed for this analysis, but the split was inspected during earlier development and this is not a preregistered test. Verdicts are SUPERIOR / EQUIVALENT / INCONCLUSIVE / WORSE, Holm-corrected. |
| Redundancy penalty and cost-awareness never active (λ = γ = 0) | Selection mode is tuned over a grid that includes active options, and a **mechanism test** forces each option on and measures its effect on the held-out test set. No vacuous ablations. |
| Retention recall is not answer accuracy | Adds a **downstream proxy**: after retention, retrieve top-k with the question and measure evidence hit@k and *answer-string-in-context*. A full-history ceiling is reported. |
| ID-prefix leak and evidence clustering inflate scores | IDs are never features. A **shortcut audit** quantifies each leak (AUC), and the primary pool is the **same-pipeline pool** (gold + other generated sessions), where style shortcuts are removed. The inflated "all sessions" pool is reported only as a reference. |
| Single benchmark, same 500 questions reused | **50/50 development/held-out split** (stratified by question type). All tuning uses dev only. Training rows that share content with held-out questions are removed (session-content dedupe). LoCoMo is an external test with a conversation-cluster bootstrap and an explicit power calculation. |

Nothing here needs an LLM once the two dataset files are present. Dataset downloads require network access if the files are not supplied locally. The held-out split was inspected during earlier development, so this is not a pristine preregistered test.

In [ ]:
import os, re, json, glob, hashlib, time, datetime, warnings, math, platform, sys
from pathlib import Path
import scipy, sklearn
import numpy as np, pandas as pd
import scipy.sparse as sp
from collections import Counter
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print

SEED = 2026
CFG = dict(
    budget=0.10,                      # fraction of pool word-count proxy that may be retained (not tokenizer tokens)
    budget_sweep=[0.05, 0.10, 0.20, 0.30],
    test_frac=0.50,                   # held-out share; not used for tuning in this notebook, but previously inspected during earlier development
    cv_folds=5,
    C_grid=[0.03, 0.3, 3.0],
    equiv_margin=0.03,                # declared up front: differences within +/-0.03 recall count as equivalent
    alpha=0.05,
    n_boot=4000,
    topk=3,                           # downstream retrieval depth
    n_random_reps=50,
    primary_pool="same_pipeline",     # declared up front
    max_questions=int(os.environ.get("MEMWEAVE_MAXQ", "0")) or None,   # smoke-test switch
    out_dir=os.environ.get("MEMWEAVE_OUT", "memweave_final_outputs"),
)
os.makedirs(CFG["out_dir"], exist_ok=True)
RNG = np.random.default_rng(SEED)
TABLES = {}          # every table is saved to out_dir at the end

PRIMARY_FAMILY = [   # pre-declared confirmatory comparisons, primary pool only, Holm over these 3
    "LR vs best simple rule (chosen on dev)",
    "LR vs recency",
    "LR vs random",
]
print("config:", {k: v for k, v in CFG.items() if k != "out_dir"})

## 1. Load data

Looks for `longmemeval_s_cleaned.json` and `locomo10.json` in common locations (Kaggle input, Colab, working dir, or the folder in `MEMWEAVE_DATA`). If they are missing it tries to download them. LoCoMo is optional: if it cannot be found, the external test is skipped.

In [ ]:
def find_file(name):
    roots = [os.environ.get("MEMWEAVE_DATA", ""), ".", "./data", "/kaggle/input", "/kaggle/working", "/content", "/mnt/user-data/uploads"]
    for r in roots:
        if r and os.path.exists(r):
            hit = glob.glob(os.path.join(r, "**", name), recursive=True)
            if hit:
                return hit[0]
    return None

def fetch(name, url):
    p = find_file(name)
    if p:
        return p
    try:
        import urllib.request
        os.makedirs("data", exist_ok=True)
        dest = os.path.join("data", name)
        print("downloading", url)
        urllib.request.urlretrieve(url, dest)
        return dest
    except Exception as e:
        print("could not obtain", name, "->", repr(e)[:120])
        return None

LME_PATH = fetch("longmemeval_s_cleaned.json",
                 "https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json")
LOCOMO_PATH = fetch("locomo10.json",
                    "https://raw.githubusercontent.com/snap-research/locomo/main/data/locomo10.json")
assert LME_PATH, "LongMemEval-S (cleaned) not found; set MEMWEAVE_DATA to the folder that contains it."
print("LongMemEval:", LME_PATH)
print("LoCoMo     :", LOCOMO_PATH)

In [ ]:
FP_RE = re.compile(r"\b(?:i|i'm|i've|i'd|i'll|my|me|mine|myself)\b", re.I)

def parse_day(s):
    m = re.match(r"\s*(\d{4})/(\d{2})/(\d{2})(?:\s*\(\w+\))?\s*(?:(\d{2}):(\d{2}))?", s)
    y, mo, d = int(m[1]), int(m[2]), int(m[3])
    hh, mm = int(m[4] or 0), int(m[5] or 0)
    return datetime.date(y, mo, d).toordinal() + (hh * 60 + mm) / 1440.0

def session_stats(turns, user_name):
    # turns: list of (speaker_role, text). Returns text and cheap, label-free session statistics.
    full = "\n".join(f"{r}: {t}" for r, t in turns)
    words = sum(len(t.split()) for _, t in turns)
    user = " ".join(t for r, t in turns if r == user_name)
    uw = len(user.split())
    fp = len(FP_RE.findall(user)) / max(uw, 1)
    dg = sum(c.isdigit() for c in user) / max(len(user), 1)
    return full, words, uw, len(turns), fp, dg

raw = json.load(open(LME_PATH))
if CFG["max_questions"]:
    raw = raw[:CFG["max_questions"]]
print("questions loaded:", len(raw))

U = dict(text=[], words=[], user=[], turns=[], fp=[], digit=[])
HIDX = {}
qrecs = []
t0 = time.time()
for qi, q in enumerate(raw):
    gold_ids = set(q["answer_session_ids"])
    qday = parse_day(q["question_date"])
    rows, hs = [], set()
    for pos, (sid, dt, sess) in enumerate(zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"])):
        turns = [(t["role"], t["content"]) for t in sess]
        key = hashlib.md5(("\n".join(f"{r}:{t}" for r, t in turns)).encode()).hexdigest()
        if key not in HIDX:
            full, w, uw, nt, fp, dg = session_stats(turns, "user")
            HIDX[key] = len(U["text"])
            U["text"].append(full); U["words"].append(w); U["user"].append(uw)
            U["turns"].append(nt); U["fp"].append(fp); U["digit"].append(dg)
        h = HIDX[key]
        gold = sid in gold_ids
        kind = "gold" if gold else ("filler" if sid.startswith(("sharegpt", "ultrachat")) else "generated")
        rows.append(dict(h=h, sid=sid, gold=gold, kind=kind, day=parse_day(dt), pos=pos / max(len(q["haystack_sessions"]) - 1, 1)))
        hs.add(h)
    qrecs.append(dict(qid=q["question_id"], qtype=q["question_type"], is_abs=q["question_id"].endswith("_abs"),
                      qday=qday, rows=rows, all_hashes=hs, question=q["question"], answer=str(q["answer"])))
for k in ("words", "user", "turns", "fp", "digit"):
    U[k] = np.asarray(U[k], dtype=np.float64)
del raw
print(f"unique sessions: {len(U['text'])}  | built in {time.time()-t0:.0f}s")

# Unsupervised sparse representation (no labels): hashed unigrams -> sublinear TF-IDF, L2-normalised.
hv = HashingVectorizer(n_features=2**19, alternate_sign=False, norm=None, lowercase=True, stop_words="english",
                       token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9']+\b", dtype=np.float32)
mats = [hv.transform(U["text"][i:i + 400]) for i in range(0, len(U["text"]), 400)]
tfidf = TfidfTransformer(sublinear_tf=True)
X = tfidf.fit_transform(sp.vstack(mats).tocsr()).astype(np.float32).tocsr()
del mats
Xq = tfidf.transform(hv.transform([q["question"] for q in qrecs])).astype(np.float32).tocsr()
print("tf-idf matrix:", X.shape, f"| {time.time()-t0:.0f}s elapsed")

## 2. Features, pools and the locked split

**Features** use only text statistics and dates. Session IDs, `has_answer` flags and the haystack position are never features. Each feature is z-scored *within the question's pool*, so the model sees relative rather than absolute values.

Four feature groups (used for ablation): `length`, `time`, `style` (user-text share, first-person rate, digit rate), `similarity` (novelty, mean similarity, top-3 similarity to other sessions).

**Pools** (IDs are used here only to define which sessions are in the evaluation pool):
- `same_pipeline` (**primary, pre-declared**): gold sessions + other synthetically generated sessions. Style shortcuts that separate evidence from ShareGPT/UltraChat filler are gone.
- `all`: the full haystack. Reported for reference; known to be inflated by artifacts.
- `filler`: gold + ShareGPT/UltraChat filler only.

Abstention (`_abs`) questions are excluded everywhere because their evidence labels do not contain the answer.

In [ ]:
FEATS = ["log_words", "n_turns", "rec_rank", "log_days_before", "user_frac", "first_person", "digit_rate",
         "novelty", "mean_sim", "top3_sim"]
GROUPS = {"length": ["log_words", "n_turns"], "time": ["rec_rank", "log_days_before"],
          "style": ["user_frac", "first_person", "digit_rate"], "similarity": ["novelty", "mean_sim", "top3_sim"]}
FI = {f: i for i, f in enumerate(FEATS)}
ALL_IDX = list(range(len(FEATS)))
TRANSFER_IDX = [FI[f] for f in FEATS if f != "log_days_before"]        # LoCoMo has no question date
ROLEFREE_IDX = [FI[f] for f in FEATS if f not in GROUPS["style"] + ["log_days_before"]]

def zscore(M):
    return (M - M.mean(0, keepdims=True)) / (M.std(0, keepdims=True) + 1e-6)

def build_features(V, words, nuser, nturns, fp, digit, days, qday):
    n = V.shape[0]
    S = (V @ V.T).toarray().astype(np.float32)
    np.fill_diagonal(S, 0.0)
    mx = S.max(1) if n > 1 else np.zeros(n)
    srt = np.sort(S, 1)
    top3 = srt[:, -min(3, n - 1):].mean(1) if n > 1 else np.zeros(n)
    order = np.argsort(np.argsort(days, kind="stable"), kind="stable")
    raw_f = np.column_stack([
        np.log1p(words), nturns, order / max(n - 1, 1),
        np.log1p(np.clip(qday - days, 0, None)) if qday is not None else np.zeros(n),
        nuser / np.maximum(words, 1), fp, digit,
        1.0 - mx, S.sum(1) / max(n - 1, 1), top3])
    return zscore(raw_f).astype(np.float32), S

POOLS = ["same_pipeline", "all", "filler"]
def pool_rows(q, pool):
    if pool == "all":
        return q["rows"]
    keep = ("gold", "generated") if pool == "same_pipeline" else ("gold", "filler")
    return [r for r in q["rows"] if r["kind"] in keep]

Q = {p: {} for p in POOLS}
for qi, q in enumerate(qrecs):
    if q["is_abs"]:
        continue
    for p in POOLS:
        rows = pool_rows(q, p)
        g = np.array([r["gold"] for r in rows])
        if len(rows) < 3 or g.all() or not g.any():
            continue
        h = np.array([r["h"] for r in rows])
        days = np.array([r["day"] for r in rows])
        F, S = build_features(X[h], U["words"][h], U["user"][h], U["turns"][h], U["fp"][h], U["digit"][h], days, q["qday"])
        Q[p][qi] = dict(h=h, F=F, S=S, gold=g, cost=U["words"][h].astype(float))

VALID = sorted(set.intersection(*[set(Q[p]) for p in POOLS]))
types = [qrecs[i]["qtype"] for i in VALID]
DEV, TEST = train_test_split(VALID, test_size=CFG["test_frac"], stratify=types, random_state=SEED)
DEV, TEST = sorted(DEV), sorted(TEST)
print({p: len(Q[p]) for p in POOLS}, "| valid in all pools:", len(VALID))
print("DEV:", len(DEV), " TEST (locked):", len(TEST))
print("mean pool size:", {p: round(np.mean([len(Q[p][i]['h']) for i in VALID]), 1) for p in POOLS})

## 3. Shortcut audit (Table 1)

How much of the old headline could be produced without understanding memory at all? Each shortcut is scored as a "gold vs not gold" detector (AUC; 0.5 = no information). None of them is used by any method below.

In [ ]:
gold_l, idp_l, reuse_l, pos_l = [], [], [], []
reuse = Counter(h for q in qrecs if not q["is_abs"] for h in q["all_hashes"])
for q in qrecs:
    if q["is_abs"]:
        continue
    for r in q["rows"]:
        gold_l.append(r["gold"]); idp_l.append(r["sid"].startswith("answer_"))
        reuse_l.append(reuse[r["h"]]); pos_l.append(r["pos"])
gold_a = np.array(gold_l)
audit = pd.DataFrame([
    ("session-ID prefix 'answer_'", roc_auc_score(gold_a, idp_l)),
    ("reuse count across questions (fewer = gold)", roc_auc_score(gold_a, -np.array(reuse_l))),
    ("position in haystack list", roc_auc_score(gold_a, pos_l)),
], columns=["shortcut", "AUC_gold_vs_rest"])

nn_rows = []
for p in POOLS:
    obs, ch = [], []
    for qi, d in Q[p].items():
        g = d["gold"]
        if g.sum() < 2:
            continue
        nn = d["S"].argmax(1)
        obs.append(g[nn[g]].mean()); ch.append((g.sum() - 1) / (len(g) - 1))
    nn_rows.append((p, len(obs), float(np.mean(obs)), float(np.mean(ch))))
nn_tab = pd.DataFrame(nn_rows, columns=["pool", "n_questions_2plus_evidence", "P(nearest neighbour of a gold session is gold)", "chance"])
TABLES["table1_shortcut_audit"] = audit
TABLES["table1b_evidence_clustering"] = nn_tab
display(audit.round(3)); display(nn_tab.round(3))

## 4. Selection, metrics and statistics

- **Budget:** retain sessions until 10% of the pool's whitespace word count is used. This is a proxy, not an exact model-token budget.
- **Selection modes:** `rank` (greedy by score), `density` (score per cost), `mmr` (greedy with a redundancy penalty λ × max similarity to already-kept sessions). Greedy fill skips items that do not fit and keeps going.
- **Metric:** evidence-session recall = share of gold sessions retained. Random is the exact expectation over many random fills.
- **Inference:** paired bootstrap over questions. Verdict rules (fixed in advance): `SUPERIOR` if CI > 0 and Holm p < 0.05; `EQUIVALENT` if the whole CI lies within ±0.03; `WORSE` if CI < 0 and Holm p < 0.05; otherwise `INCONCLUSIVE`.

In [ ]:
def select_idx(score, cost, S, budget, mode="rank", lam=0.0):
    B = budget * cost.sum()
    n = len(score)
    chosen, used = [], 0.0
    if mode == "mmr":
        avail, maxsim = set(range(n)), np.zeros(n)
        while avail:
            best, bv = None, -1e18
            for i in avail:
                if used + cost[i] > B:
                    continue
                v = score[i] - lam * maxsim[i]
                if v > bv:
                    bv, best = v, i
            if best is None:
                break
            chosen.append(best); used += cost[best]; avail.remove(best)
            maxsim = np.maximum(maxsim, S[best])
        return np.array(chosen, dtype=int)
    key = score if mode == "rank" else score / np.maximum(cost, 1.0)
    for i in np.argsort(-key, kind="stable"):
        if used + cost[i] <= B:
            chosen.append(i); used += cost[i]
    return np.array(chosen, dtype=int)

def recall_of(d, chosen):
    return d["gold"][chosen].sum() / d["gold"].sum() if len(chosen) else 0.0

def selections(pool, qis, scores, budget=None, mode="rank", lam=0.0):
    budget = CFG["budget"] if budget is None else budget
    return {qi: select_idx(scores[qi], Q[pool][qi]["cost"], Q[pool][qi]["S"], budget, mode, lam) for qi in qis}

def recalls(pool, qis, sel):
    return np.array([recall_of(Q[pool][qi], sel[qi]) for qi in qis])

def random_selections(pool, qis, budget=None, reps=None, seed=0):
    budget = CFG["budget"] if budget is None else budget
    reps = reps or CFG["n_random_reps"]
    rng = np.random.default_rng(seed)
    out = {}
    for qi in qis:
        d = Q[pool][qi]; n = len(d["cost"])
        out[qi] = [select_idx(rng.random(n), d["cost"], d["S"], budget) for _ in range(reps)]
    return out

def random_recalls(pool, qis, rsel):
    return np.array([np.mean([recall_of(Q[pool][qi], c) for c in rsel[qi]]) for qi in qis])

SIMPLE = {   # label-free heuristics; all computed from the pool's own features
    "recency":         lambda F: F[:, FI["rec_rank"]],
    "longest":         lambda F: F[:, FI["log_words"]],
    "high_centrality": lambda F: F[:, FI["mean_sim"]],
    "low_novelty":     lambda F: -F[:, FI["novelty"]],
}
def simple_scores(pool, qis, name):
    return {qi: SIMPLE[name](Q[pool][qi]["F"]) for qi in qis}

# ---- statistics
def boot_ci(x, B=None, seed=1):
    B = B or CFG["n_boot"]
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), (B, len(x)))
    m = x[idx].mean(1)
    return float(np.mean(x)), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def paired(a, b, B=None, seed=2):
    """Paired percentile-bootstrap CI plus a null-centred sign-flip randomization p-value.

    The sign-flip test assumes method labels are exchangeable within question under
    the null. Questions are treated as the resampling units; shared-session dependence
    remains a limitation and is disclosed in the notebook.
    """
    B = B or CFG["n_boot"]
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if len(d) == 0 or not np.isfinite(d).all():
        raise ValueError("paired() requires non-empty, finite paired observations")
    rng = np.random.default_rng(seed)
    m = d[rng.integers(0, len(d), (B, len(d)))].mean(1)
    signs = rng.choice(np.array([-1.0, 1.0]), size=(B, len(d)))
    null_means = (signs * d).mean(1)
    p = (1 + np.count_nonzero(np.abs(null_means) >= abs(d.mean()))) / (B + 1)
    return dict(diff=float(d.mean()), ci_low=float(np.percentile(m, 2.5)),
                ci_high=float(np.percentile(m, 97.5)), p=float(p))

def holm(ps):
    ps = np.asarray(ps, float); m = len(ps); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(np.argsort(ps)):
        run = max(run, (m - rank) * ps[i]); adj[i] = min(1.0, run)
    return adj

def verdict(row, margin=None):
    margin = CFG["equiv_margin"] if margin is None else margin
    if row["ci_low"] > 0 and row["p_holm"] < CFG["alpha"]:
        return "SUPERIOR"
    if row["ci_high"] < 0 and row["p_holm"] < CFG["alpha"]:
        return "WORSE"
    if row["ci_low"] >= -margin and row["ci_high"] <= margin:
        return "EQUIVALENT (within +/-%.2f)" % margin
    return "INCONCLUSIVE"

def compare_table(pairs, family_name):
    rows = [dict(comparison=name, **paired(a, b)) for name, a, b in pairs]
    ph = holm([r["p"] for r in rows])
    for r, p in zip(rows, ph):
        r["p_holm"] = float(p); r["verdict"] = verdict(r); r["family"] = family_name
    return pd.DataFrame(rows)
print("helpers ready")

## 5. Tuning on DEV only

For each pool: pick the logistic-regression regularisation `C` by 5-fold CV on dev, drop feature groups that do not help (backward elimination on dev CV), then pick the selection mode with a **one-standard-error rule** (simplest mode within 1 SE of the best; order: rank < density < mmr with larger λ). The best *simple* rule is also chosen on dev. Training rows whose session content appears in any held-out question are dropped, so a fold cannot learn from sessions it will be scored on.

In [ ]:
def stack(pool, qis, idx):
    Xs = np.vstack([Q[pool][q]["F"][:, idx] for q in qis])
    ys = np.concatenate([Q[pool][q]["gold"] for q in qis]).astype(int)
    hs = np.concatenate([Q[pool][q]["h"] for q in qis])
    return Xs, ys, hs

def make_model(kind, C):
    if kind == "hgb":
        return HistGradientBoostingClassifier(max_depth=3, max_iter=120, learning_rate=0.08, random_state=0)
    return LogisticRegression(C=C, class_weight="balanced", max_iter=500)

def fit_scores(pool, train_qis, test_qis, C, idx=ALL_IDX, kind="lr"):
    held = np.fromiter(set().union(*[qrecs[q]["all_hashes"] for q in test_qis]), dtype=np.int64)
    Xtr, ytr, htr = stack(pool, train_qis, idx)
    keep = ~np.isin(htr, held)
    m = make_model(kind, C).fit(Xtr[keep], ytr[keep])
    out = {}
    for q in test_qis:
        out[q] = m.predict_proba(Q[pool][q]["F"][:, idx])[:, 1]
    return out, int(keep.sum()), int(len(keep))

SEL_OPTIONS = [("rank", 0.0), ("density", 0.0), ("mmr", 0.1), ("mmr", 0.25), ("mmr", 0.5)]
rng = np.random.default_rng(SEED)
perm = rng.permutation(DEV)
FOLDS = [sorted(f.tolist()) for f in np.array_split(perm, CFG["cv_folds"])]

def idx_of(groups):
    cols = [f for g in groups for f in GROUPS[g]]
    return [FI[f] for f in FEATS if f in cols]

def oof_scores(pool, idx, C):
    out = {}
    for k, test_f in enumerate(FOLDS):
        train_f = [q for j, f in enumerate(FOLDS) if j != k for q in f]
        sc, _, _ = fit_scores(pool, train_f, test_f, C, idx=idx)
        out.update(sc)
    return out

def cv_recall(pool, idx, C):
    return recalls(pool, DEV, selections(pool, DEV, oof_scores(pool, idx, C))).mean()

TUNED, TUNE_ROWS, BEST_SIMPLE = {}, [], {}
for pool in POOLS:
    # (a) regularisation
    cvC = {C: cv_recall(pool, ALL_IDX, C) for C in CFG["C_grid"]}
    Cb = max(cvC, key=cvC.get)
    for C, v in cvC.items():
        TUNE_ROWS.append((pool, "C", C, v, np.nan))
    # (b) backward elimination of feature groups, on dev CV only (drop a group if recall does not get worse)
    groups = list(GROUPS)
    cur = cv_recall(pool, idx_of(groups), Cb)
    while len(groups) > 1:
        trials = [(cv_recall(pool, idx_of([x for x in groups if x != g]), Cb), g) for g in groups]
        m_, g_ = max(trials)
        if m_ >= cur:
            groups.remove(g_); cur = m_
            TUNE_ROWS.append((pool, "group_removed", g_, m_, np.nan))
        else:
            break
    idx = idx_of(groups)
    # (c) selection mode on the final feature set, one-standard-error rule
    oof = oof_scores(pool, idx, Cb)
    opt_stats = []
    for mode, lam in SEL_OPTIONS:
        r = recalls(pool, DEV, selections(pool, DEV, oof, mode=mode, lam=lam))
        opt_stats.append((mode, lam, r.mean(), r.std(ddof=1) / np.sqrt(len(r))))
    best = max(o[2] for o in opt_stats)
    se_best = [o for o in opt_stats if o[2] == best][0][3]
    chosen = next(o for o in opt_stats if o[2] >= best - se_best)
    TUNED[pool] = dict(C=Cb, mode=chosen[0], lam=chosen[1], groups=groups, idx=idx)
    for mode, lam, m_, se_ in opt_stats:
        TUNE_ROWS.append((pool, f"selection:{mode}", lam, m_, se_))
    # (d) best simple rule on dev
    simple_dev = {n: recalls(pool, DEV, selections(pool, DEV, simple_scores(pool, DEV, n))).mean() for n in SIMPLE}
    BEST_SIMPLE[pool] = max(simple_dev, key=simple_dev.get)
    TUNE_ROWS.extend((pool, "simple_rule", n, v, np.nan) for n, v in simple_dev.items())
tune_df = pd.DataFrame(TUNE_ROWS, columns=["pool", "what", "value", "dev_cv_recall", "se"])
TABLES["table2_dev_tuning"] = tune_df
print("tuned config :", {p: {k: v for k, v in c.items() if k != "idx"} for p, c in TUNED.items()})
print("best simple  :", BEST_SIMPLE)
display(tune_df.round(3))

## 6. held-out test: main results (Tables 3 and 4)

The test split is evaluated once, with configurations fixed in section 5. The **primary family** (3 comparisons on the primary pool, Holm-corrected) is the only confirmatory analysis. Everything else is labelled secondary or descriptive.

In [ ]:
RES, SEL = {}, {}
for pool in POOLS:
    cfg = TUNED[pool]
    sc, kept, tot = fit_scores(pool, DEV, TEST, cfg["C"], idx=cfg["idx"])
    sc_all, _, _ = fit_scores(pool, DEV, TEST, cfg["C"], idx=ALL_IDX)
    sel_lr = selections(pool, TEST, sc, mode=cfg["mode"], lam=cfg["lam"])
    rsel = random_selections(pool, TEST)
    bs = BEST_SIMPLE[pool]
    sel_simple = {n: selections(pool, TEST, simple_scores(pool, TEST, n)) for n in SIMPLE}
    RES[pool] = {"random": random_recalls(pool, TEST, rsel),
                 "recency": recalls(pool, TEST, sel_simple["recency"]),
                 "longest": recalls(pool, TEST, sel_simple["longest"]),
                 "high_centrality": recalls(pool, TEST, sel_simple["high_centrality"]),
                 "low_novelty": recalls(pool, TEST, sel_simple["low_novelty"]),
                 "best_simple(dev-chosen)": recalls(pool, TEST, sel_simple[bs]),
                 "LR (tuned)": recalls(pool, TEST, sel_lr),
                 "LR (all groups)": recalls(pool, TEST, selections(pool, TEST, sc_all, mode=cfg["mode"], lam=cfg["lam"]))}
    SEL[pool] = dict(lr=sel_lr, random=rsel, simple=sel_simple, scores=sc)
    print(f"{pool}: trained on {kept}/{tot} rows after content de-duplication")

rows = []
for pool in POOLS:
    for m, arr in RES[pool].items():
        mean, lo, hi = boot_ci(arr)
        rows.append((pool, m, mean, lo, hi, len(arr)))
main_tab = pd.DataFrame(rows, columns=["pool", "method", "recall@10%", "ci_low", "ci_high", "n_test_questions"])
TABLES["table3_main_results_heldout"] = main_tab
display(main_tab.pivot(index="method", columns="pool", values="recall@10%").round(3))

P = CFG["primary_pool"]; R = RES[P]
prim = compare_table([
    (PRIMARY_FAMILY[0], R["LR (tuned)"], R["best_simple(dev-chosen)"]),
    (PRIMARY_FAMILY[1], R["LR (tuned)"], R["recency"]),
    (PRIMARY_FAMILY[2], R["LR (tuned)"], R["random"]),
], "PRIMARY (confirmatory)")
prim.insert(0, "pool", P)
desc = []
for pool in POOLS:
    if pool == P:
        continue
    t = compare_table([("LR vs best simple rule (chosen on dev)", RES[pool]["LR (tuned)"], RES[pool]["best_simple(dev-chosen)"]),
                       ("LR vs random", RES[pool]["LR (tuned)"], RES[pool]["random"])], "descriptive")
    t.insert(0, "pool", pool); desc.append(t)
cmp_tab = pd.concat([prim] + desc, ignore_index=True)
TABLES["table4_primary_and_descriptive_comparisons"] = cmp_tab
print("best simple rule on", P, "=", BEST_SIMPLE[P], "| LR config:", {k: v for k, v in TUNED[P].items() if k != "idx"})
display(cmp_tab.round(4))

pw = RES[P]["LR (tuned)"] - RES[P]["best_simple(dev-chosen)"]
sd = pw.std(ddof=1); n = len(pw)
MDE = (1.96 + 0.84) * sd / math.sqrt(n)
print(f"Power: SD of paired per-question difference = {sd:.3f}, n = {n} -> minimum detectable difference (80% power) ~ {MDE:.3f}")
print("A true gain of 0.03 recall is", "detectable" if MDE <= 0.03 else "NOT reliably detectable", "with this test set.")

## 7. Ablations, mechanism tests and budget sweep (Tables 5 to 7)

- **Group ablation:** remove one feature group, retrain, and compare with the full model on the held-out test set (Holm across groups). A group "contributes" only if the CI excludes 0 *and* Holm p < 0.05.
- **Mechanism test:** force the redundancy-aware and cost-aware selectors on (they were never active in v2.2) and measure their effect against plain rank selection.
- **Model class:** gradient boosting versus logistic regression.
- **Budget sweep:** LR vs the dev-chosen simple rule at 5/10/20/30%, Holm across budgets.

In [ ]:
cfgP = TUNED[P]
full = RES[P]["LR (tuned)"]

# group ablation
abl_pairs = []
for g in (cfgP["groups"] if len(cfgP["groups"]) > 1 else []):
    idx = idx_of([x for x in cfgP["groups"] if x != g])
    sc, _, _ = fit_scores(P, DEV, TEST, cfgP["C"], idx=idx)
    r = recalls(P, TEST, selections(P, TEST, sc, mode=cfgP["mode"], lam=cfgP["lam"]))
    abl_pairs.append((f"remove group: {g}", full, r))
abl = compare_table(abl_pairs, "ablation")
# diff = full model minus model without the group: positive means the group helps
abl["contributes"] = np.where((abl["ci_low"] > 0) & (abl["p_holm"] < CFG["alpha"]), "YES (group helps)",
                      np.where((abl["ci_high"] < 0) & (abl["p_holm"] < CFG["alpha"]), "NO (removing it IMPROVES recall)", "no reliable contribution"))
TABLES["table5_group_ablation"] = abl
display(abl.round(4))

# mechanism test: force each selector on, compare to plain rank selection with the same scores
sc_main = SEL[P]["scores"]
rank_r = recalls(P, TEST, selections(P, TEST, sc_main, mode="rank"))
mech_pairs = []
for mode, lam in [("density", 0.0), ("mmr", 0.25), ("mmr", 0.5)]:
    r = recalls(P, TEST, selections(P, TEST, sc_main, mode=mode, lam=lam))
    mech_pairs.append((f"{mode}{'' if mode != 'mmr' else ' lambda=' + str(lam)} minus rank", r, rank_r))
mech = compare_table(mech_pairs, "mechanism")
TABLES["table6_mechanism_test"] = mech
display(mech.round(4))

# model class
sc_h, _, _ = fit_scores(P, DEV, TEST, cfgP["C"], idx=cfgP["idx"], kind="hgb")
hgb_r = recalls(P, TEST, selections(P, TEST, sc_h, mode=cfgP["mode"], lam=cfgP["lam"]))
mc = compare_table([("gradient boosting minus LR", hgb_r, full),
                    ("LR with dev-selected groups minus LR with all groups", full, RES[P]["LR (all groups)"])], "model class")
TABLES["table6b_model_class"] = mc
display(mc.round(4))

# budget sweep
sweep_pairs = []
for b in CFG["budget_sweep"]:
    lr_b = recalls(P, TEST, selections(P, TEST, sc_main, budget=b, mode=cfgP["mode"], lam=cfgP["lam"]))
    sm_b = recalls(P, TEST, selections(P, TEST, simple_scores(P, TEST, BEST_SIMPLE[P]), budget=b))
    sweep_pairs.append((f"budget {int(b*100)}%: LR minus {BEST_SIMPLE[P]}", lr_b, sm_b))
sweep = compare_table(sweep_pairs, "budget sweep")
TABLES["table7_budget_sweep"] = sweep
display(sweep.round(4))

**Split-seed robustness (primary pool).** The whole dev-tuning and held-out procedure is repeated on 8 other random splits (rank selection only, to keep it fast). The held-out halves overlap, so this is *not* independent replication; it only shows whether the verdict depends on one particular split.

In [ ]:
def robustness(seeds):
    global DEV, FOLDS
    saved = (DEV, FOLDS)
    rows = []
    for sd in seeds:
        dev, test = train_test_split(VALID, test_size=CFG["test_frac"], stratify=types, random_state=sd)
        dev, test = sorted(dev), sorted(test)
        pm = np.random.default_rng(sd).permutation(dev)
        DEV = dev
        FOLDS = [sorted(f.tolist()) for f in np.array_split(pm, CFG["cv_folds"])]
        cvC = {C: cv_recall(P, ALL_IDX, C) for C in CFG["C_grid"]}
        Cb = max(cvC, key=cvC.get)
        groups = list(GROUPS); cur = cv_recall(P, idx_of(groups), Cb)
        while len(groups) > 1:
            trials = [(cv_recall(P, idx_of([x for x in groups if x != g]), Cb), g) for g in groups]
            m_, g_ = max(trials)
            if m_ >= cur:
                groups.remove(g_); cur = m_
            else:
                break
        sc, _, _ = fit_scores(P, dev, test, Cb, idx=idx_of(groups))
        lr = recalls(P, test, selections(P, test, sc))
        simple = {n: recalls(P, dev, selections(P, dev, simple_scores(P, dev, n))).mean() for n in SIMPLE}
        bs = max(simple, key=simple.get)
        sm = recalls(P, test, selections(P, test, simple_scores(P, test, bs)))
        pr = paired(lr, sm)
        rows.append((sd, "+".join(groups), bs, lr.mean(), sm.mean(), pr["diff"], pr["ci_low"], pr["ci_high"]))
    DEV, FOLDS = saved
    return pd.DataFrame(rows, columns=["split_seed", "groups_kept", "best_simple", "LR_recall", "simple_recall", "diff", "ci_low", "ci_high"])

rob = robustness([11, 12, 13, 14, 15, 16, 17, 18])
TABLES["table7b_split_seed_robustness"] = rob
display(rob.round(3))
ROB = dict(n=len(rob), pos=int((rob["diff"] > 0).sum()), sig=int((rob["ci_low"] > 0).sum()), med=float(rob["diff"].median()))
print(f"LR minus best simple rule over {ROB['n']} re-splits: median {ROB['med']:+.3f}; positive in {ROB['pos']}/{ROB['n']}; 95% CI excludes 0 (uncorrected) in {ROB['sig']}/{ROB['n']}")

## 8. Downstream proxy: does retention help answering? (Table 8)

Retention recall is not answer accuracy. After retention, we retrieve the top-k retained sessions with the *question* (TF-IDF cosine) and measure:
- **evidence hit@k:** a gold session is in the top-k;
- **answer-in-context:** the gold answer string appears in a retrieved session. Only questions with a short answer (≤ 40 characters) that literally appears in a gold session are eligible, so the metric is well-defined without an LLM.

`full_history` is the ceiling (retrieve from everything, no memory limit). This is still a proxy, not LLM answer accuracy; see the limitations.

In [ ]:
K = CFG["topk"]
def eligible(qi):
    q = qrecs[qi]; a = q["answer"].lower().strip()
    if not a or len(a) > 40:
        return False
    return any(a in U["text"][r["h"]].lower() for r in q["rows"] if r["gold"])
ELIG = {qi: eligible(qi) for qi in TEST}
print("answer-bearing eligible test questions:", sum(ELIG.values()), "of", len(TEST))

def downstream(pool, sel_map, qis):
    hit, ans = [], []
    for qi in qis:
        d = Q[pool][qi]
        sims = (X[d["h"]] @ Xq[qi].T).toarray().ravel()
        sels = sel_map[qi] if isinstance(sel_map[qi], list) else [sel_map[qi]]
        hs, as_ = [], []
        for ch in sels:
            ch = np.asarray(ch, dtype=int)
            top = ch[np.argsort(-sims[ch], kind="stable")[:K]] if len(ch) else ch
            hs.append(float(d["gold"][top].any()) if len(top) else 0.0)
            if ELIG[qi]:
                a = qrecs[qi]["answer"].lower().strip()
                as_.append(float(any(a in U["text"][d["h"][j]].lower() for j in top)) if len(top) else 0.0)
        hit.append(np.mean(hs)); ans.append(np.mean(as_) if as_ else np.nan)
    return np.array(hit), np.array(ans)

DS_ROWS, DS_CMP = [], []
for pool in [P, "all"]:
    bs = BEST_SIMPLE[pool]
    methods = {"random": SEL[pool]["random"], "recency": SEL[pool]["simple"]["recency"],
               f"best_simple ({bs})": SEL[pool]["simple"][bs], "LR (tuned)": SEL[pool]["lr"],
               "full_history (ceiling)": {qi: np.arange(len(Q[pool][qi]["h"])) for qi in TEST}}
    out = {m: downstream(pool, s, TEST) for m, s in methods.items()}
    el = np.array([ELIG[q] for q in TEST])
    for m, (h_, a_) in out.items():
        DS_ROWS.append((pool, m, h_.mean(), np.nanmean(a_[el]) if el.any() else np.nan))
    lr_h, lr_a = out["LR (tuned)"]
    for other in ["recency", f"best_simple ({bs})", "random"]:
        oh, oa = out[other]
        ph = paired(lr_h, oh); pa = paired(lr_a[el], oa[el]) if el.any() else dict(diff=np.nan, ci_low=np.nan, ci_high=np.nan, p=np.nan)
        DS_CMP.append((pool, f"LR minus {other}", ph["diff"], ph["ci_low"], ph["ci_high"], pa["diff"], pa["ci_low"], pa["ci_high"]))
ds = pd.DataFrame(DS_ROWS, columns=["pool", "method", f"evidence_hit@{K}", f"answer_in_context@{K}"])
dsc = pd.DataFrame(DS_CMP, columns=["pool", "comparison", "hit_diff", "hit_lo", "hit_hi", "ans_diff", "ans_lo", "ans_hi"])
TABLES["table8_downstream_proxy"] = ds
TABLES["table8b_downstream_paired"] = dsc
display(ds.round(3)); display(dsc.round(3))

## 9. External test on LoCoMo (Table 9)

The selector is trained on LongMemEval only, with the configuration fixed on LongMemEval dev. Nothing is tuned on LoCoMo. Features that need a question date are dropped. Because two features depend on which speaker is the "user" (an arbitrary convention in LoCoMo), a **role-free** variant without the `style` group is also run. With only 10 conversations the test has limited power; the minimum detectable difference is printed.

In [ ]:
# Preserve the path returned by fetch(); only override it when LOCOMO_DATA is explicit.
_loco_override = os.environ.get("LOCOMO_DATA")
if _loco_override:
    LOCOMO_PATH = Path(_loco_override)
elif LOCOMO_PATH:
    LOCOMO_PATH = Path(LOCOMO_PATH)
else:
    LOCOMO_PATH = None
if LOCOMO_PATH is not None and not LOCOMO_PATH.is_file():
    print("LoCoMo file not found:", LOCOMO_PATH, "| external test will be skipped.")
    LOCOMO_PATH = None

LOCO = None
if LOCOMO_PATH is not None:
    data = json.load(open(LOCOMO_PATH))
    convs = []
    for c in data:
        conv = c["conversation"]; a_name = conv.get("speaker_a")
        nums = sorted(int(k.split("_")[1]) for k in conv if re.fullmatch(r"session_\d+", k))
        texts, W, UW, NT, FPv, DG = [], [], [], [], [], []
        for n in nums:
            turns = [(("user" if t["speaker"] == a_name else "assistant"), t.get("text", "")) for t in conv[f"session_{n}"]]
            full, w, uw, nt, fp, dg = session_stats(turns, "user")
            texts.append(full); W.append(w); UW.append(uw); NT.append(nt); FPv.append(fp); DG.append(dg)
        qs = []
        for qa in c["qa"]:
            if qa.get("category") == 5:
                continue
            ev = {int(m) for e in qa.get("evidence", []) for m in re.findall(r"D(\d+):", str(e))}
            ev = {nums.index(e) for e in ev if e in nums}
            if ev:
                qs.append(ev)
        V = tfidf.transform(hv.transform(texts)).astype(np.float32).tocsr()
        F, S = build_features(V, np.array(W, float), np.array(UW, float), np.array(NT, float), np.array(FPv), np.array(DG),
                              np.arange(len(nums), dtype=float), None)
        convs.append(dict(F=F, S=S, cost=np.array(W, float), qs=qs, n=len(nums)))
    LOCO = convs
    print("LoCoMo conversations:", len(convs), "| questions with evidence:", sum(len(c["qs"]) for c in convs))
else:
    print("LoCoMo file not found: external test skipped.")

In [ ]:
LOCO_RES = {}
if LOCO:
    allq = [q for q in VALID]
    cfgP = TUNED[P]
    models = {}
    base_idx = [i for i in cfgP["idx"] if i in TRANSFER_IDX] or TRANSFER_IDX
    role_idx = [i for i in base_idx if FEATS[i] not in GROUPS["style"]] or ROLEFREE_IDX
    for name, idx in [("LR (transfer)", base_idx), ("LR (role-free)", role_idx)]:
        Xtr, ytr, _ = stack(P, allq, idx)
        models[name] = (make_model("lr", cfgP["C"]).fit(Xtr, ytr), idx)

    def conv_recalls(score_fn, mode="rank", lam=0.0, reps=None):
        per_conv = []
        for c in LOCO:
            if reps:
                rng = np.random.default_rng(0)
                sels = [select_idx(rng.random(c["n"]), c["cost"], c["S"], CFG["budget"]) for _ in range(reps)]
            else:
                sels = [select_idx(score_fn(c), c["cost"], c["S"], CFG["budget"], mode, lam)]
            rs = []
            for ev in c["qs"]:
                rs.append(np.mean([len(set(s.tolist()) & ev) / len(ev) for s in sels]))
            per_conv.append(np.mean(rs))
        return np.array(per_conv)

    LOCO_RES["random"] = conv_recalls(None, reps=CFG["n_random_reps"])
    for n_ in ["recency", "low_novelty", "high_centrality"]:
        LOCO_RES[n_] = conv_recalls(lambda c, f=SIMPLE[n_]: f(c["F"]))
    for name, (m, idx) in models.items():
        LOCO_RES[name] = conv_recalls(lambda c, m=m, idx=idx: m.predict_proba(c["F"][:, idx])[:, 1], cfgP["mode"], cfgP["lam"])

    def cluster_boot(a, b, B=4000, seed=3):
        d = a - b; rng = np.random.default_rng(seed)
        m = d[rng.integers(0, len(d), (B, len(d)))].mean(1)
        return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

    rows = []
    for name in ["LR (transfer)", "LR (role-free)"]:
        for other in ["recency", "low_novelty", "random"]:
            dmean, lo, hi = cluster_boot(LOCO_RES[name], LOCO_RES[other])
            rows.append((name, other, LOCO_RES[name].mean(), LOCO_RES[other].mean(), dmean, lo, hi))
    loc_tab = pd.DataFrame(rows, columns=["method", "vs", "method_recall", "other_recall", "diff", "ci_low", "ci_high"])
    sdc = (LOCO_RES["LR (transfer)"] - LOCO_RES["low_novelty"]).std(ddof=1)
    LOCO_MDE = (1.96 + 0.84) * sdc / math.sqrt(len(LOCO))
    TABLES["table9_locomo_external"] = loc_tab
    display(loc_tab.round(4))
    print(f"LoCoMo power: SD across conversations = {sdc:.3f}, n = {len(LOCO)} -> minimum detectable difference ~ {LOCO_MDE:.3f}")

## 10. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
order = ["random", "recency", "longest", "high_centrality", "low_novelty", "LR (tuned)"]
w = 0.26
for k, pool in enumerate(POOLS):
    t = main_tab[main_tab.pool == pool].set_index("method").loc[order]
    axes[0].bar(np.arange(len(order)) + (k - 1) * w, t["recall@10%"], w, label=pool,
                yerr=[t["recall@10%"] - t["ci_low"], t["ci_high"] - t["recall@10%"]], capsize=2)
axes[0].set_xticks(np.arange(len(order))); axes[0].set_xticklabels(order, rotation=30, ha="right")
axes[0].set_ylabel("evidence-session recall at 10% budget"); axes[0].set_title("held-out test: same method, three pools")
axes[0].legend(title="pool")

sw = TABLES["table7_budget_sweep"]
xs = [int(b * 100) for b in CFG["budget_sweep"]]
axes[1].errorbar(xs, sw["diff"], yerr=[sw["diff"] - sw["ci_low"], sw["ci_high"] - sw["diff"]], marker="o", capsize=3)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].axhspan(-CFG["equiv_margin"], CFG["equiv_margin"], color="gray", alpha=0.15, label="equivalence margin")
axes[1].set_xlabel("budget (% of pool tokens)"); axes[1].set_ylabel("LR minus best simple rule (recall)")
axes[1].set_title(f"Primary pool ({P}): paired difference"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CFG["out_dir"], "fig_main_results.png"), dpi=150)
plt.show()

## 11. Auto-generated findings, and what you may claim

Everything below is computed from the tables above. The resume paragraph changes automatically with the verdicts, so it cannot overstate the result.

In [ ]:
def get(df, name):
    return df[df["comparison"] == name].iloc[0]

L = []
L.append(f"MemWeave-Final | primary pool = {P} | budget {int(CFG['budget']*100)}% | DEV {len(DEV)} / held-out test {len(TEST)} questions | equivalence margin +/-{CFG['equiv_margin']}")
L.append("")
L.append("SHORTCUT AUDIT")
for _, r in TABLES["table1_shortcut_audit"].iterrows():
    L.append(f"  {r['shortcut']}: AUC {r['AUC_gold_vs_rest']:.3f}")
for _, r in TABLES["table1b_evidence_clustering"].iterrows():
    L.append(f"  [{r['pool']}] nearest neighbour of a gold session is gold: {r.iloc[2]:.3f} (chance {r['chance']:.3f})")
L.append("")
L.append("PRIMARY FAMILY (held-out test, Holm over 3)")
for _, r in prim.iterrows():
    L.append(f"  {r['comparison']}: diff {r['diff']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}], Holm p={r['p_holm']:.3f} -> {r['verdict']}")
L.append(f"  Best simple rule chosen on dev: {BEST_SIMPLE[P]} | LR config: C={TUNED[P]['C']}, selection={TUNED[P]['mode']}(lambda={TUNED[P]['lam']}), feature groups kept on dev: {TUNED[P]['groups']}")
L.append(f"  Minimum detectable difference (80% power): {MDE:.3f}")
L.append(f"  Split-seed robustness: median diff {ROB['med']:+.3f}; positive in {ROB['pos']}/{ROB['n']} re-splits; CI excludes 0 (uncorrected) in {ROB['sig']}/{ROB['n']}")
L.append("")
L.append("POOL DEPENDENCE (LR minus best simple rule, descriptive)")
for _, r in cmp_tab[(cmp_tab.family == 'descriptive') & cmp_tab.comparison.str.contains('best simple')].iterrows():
    L.append(f"  {r['pool']}: diff {r['diff']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}] -> {r['verdict']}")
L.append(f"  {P}: diff {get(prim, PRIMARY_FAMILY[0])['diff']:+.3f}")
L.append("")
L.append("COMPONENTS")
for _, r in abl.iterrows():
    L.append(f"  {r['comparison']}: {r['diff']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}] -> {r['contributes']}")
for _, r in mech.iterrows():
    L.append(f"  mechanism: {r['comparison']}: {r['diff']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}] -> {r['verdict']}")
L.append("")
L.append("DOWNSTREAM PROXY")
for _, r in ds.iterrows():
    L.append(f"  [{r['pool']}] {r['method']}: hit@{K} {r.iloc[2]:.3f} | answer-in-context@{K} {r.iloc[3]:.3f}")
if LOCO:
    L.append("")
    L.append("LOCOMO EXTERNAL (conversation-cluster bootstrap)")
    for _, r in TABLES["table9_locomo_external"].iterrows():
        L.append(f"  {r['method']} vs {r['vs']}: {r['diff']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]")
    _lr, _rd = LOCO_RES["LR (transfer)"].mean(), LOCO_RES["random"].mean()
    _d, _lo, _hi = cluster_boot(LOCO_RES["LR (transfer)"], LOCO_RES["random"])
    LOCO_TRANSFERS = _lo > 0
    L.append(f"  LR recall {_lr:.3f} vs random (chance) {_rd:.3f}: diff {_d:+.3f} [{_lo:+.3f}, {_hi:+.3f}] -> " + ("above chance" if LOCO_TRANSFERS else "NOT distinguishable from chance"))
    L.append(f"  minimum detectable difference ~ {LOCO_MDE:.3f}")
findings = "\n".join(L)
print(findings)

# ---- resume / abstract paragraph that follows the verdicts
a = audit.set_index("shortcut")["AUC_gold_vs_rest"]
nn_all = nn_tab[nn_tab.pool == "all"].iloc[0]
v0 = get(prim, PRIMARY_FAMILY[0])
if v0["verdict"] == "SUPERIOR":
    outcome = f"the learned selector beat the best simple rule on a held-out test set by {v0['diff']:+.3f} recall (95% CI [{v0['ci_low']:+.3f}, {v0['ci_high']:+.3f}], Holm-corrected)"
elif v0["verdict"].startswith("EQUIVALENT"):
    outcome = f"the learned selector was statistically equivalent (within +/-{CFG['equiv_margin']}) to the best simple rule on a held-out test set (diff {v0['diff']:+.3f}, 95% CI [{v0['ci_low']:+.3f}, {v0['ci_high']:+.3f}])"
elif v0["verdict"] == "WORSE":
    outcome = f"the learned selector underperformed the best simple rule on a held-out test set ({v0['diff']:+.3f}, 95% CI [{v0['ci_low']:+.3f}, {v0['ci_high']:+.3f}])"
else:
    outcome = f"the learned selector showed no reliable difference from the best simple rule on a held-out test set (diff {v0['diff']:+.3f}, 95% CI [{v0['ci_low']:+.3f}, {v0['ci_high']:+.3f}]; minimum detectable difference ~{MDE:.2f})"
rob_sentence = f" Across {ROB['n']} alternative splits the gain was positive in {ROB['pos']} and CI-excluding-zero in {ROB['sig']}."
loco_sentence = ""
if LOCO:
    if LOCO_TRANSFERS:
        loco_sentence = f" On the external LoCoMo test the selector was above chance ({_lr:.3f} vs {_rd:.3f} random)."
    else:
        loco_sentence = f" The selector did not transfer to LoCoMo (recall {_lr:.3f} vs {_rd:.3f} random; minimum detectable difference ~{LOCO_MDE:.2f})."
resume = (
    "Built and rigorously audited a budget-constrained long-term memory retention pipeline on LongMemEval-S"
    + (" and LoCoMo" if LOCO else "") + ". "
    f"Identified evaluation artifacts: session-ID prefix alone separates evidence sessions (AUC {a.iloc[0]:.2f}), and evidence sessions are each other's nearest neighbour "
    f"{nn_all.iloc[2]*100:.0f}% of the time vs {nn_all['chance']*100:.0f}% by chance. "
    f"After controlling for them (same-pipeline distractor pool, content-deduplicated training, locked {len(TEST)}-question test set, pre-declared Holm-corrected comparisons), {outcome}." + rob_sentence + loco_sentence
)
if CFG["max_questions"]:
    print("\nSMOKE TEST ONLY: MEMWEAVE_MAXQ is set; do not report these metrics as benchmark results.")
else:
    print("\nRESUME / ABSTRACT PARAGRAPH:\n" + resume)

for name, df in TABLES.items():
    df.to_csv(os.path.join(CFG["out_dir"], name + ".csv"), index=False)
open(os.path.join(CFG["out_dir"], "findings.txt"), "w").write(findings + "\n\nRESUME / ABSTRACT PARAGRAPH:\n" + resume + "\n")
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def dataset_record(path):
    if path is None:
        return None
    p = os.fspath(path)
    if not os.path.isfile(p):
        return {"path": p, "exists": False}
    return {"path": os.path.abspath(p), "exists": True,
            "bytes": os.path.getsize(p), "sha256": sha256_file(p)}

run_manifest = {
    "project": "MemWeave",
    "evaluation_status": "held_out_post_hoc_not_preregistered",
    "run_mode": "smoke_test" if CFG["max_questions"] else "full_dataset",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "datasets": {
        "longmemeval_s_cleaned": dataset_record(LME_PATH),
        "locomo10": dataset_record(LOCOMO_PATH),
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "config": CFG,
    "tuned": TUNED,
    "best_simple": BEST_SIMPLE,
    "n_questions_loaded": len(qrecs),
    "n_valid_questions": len(VALID),
    "n_dev": len(DEV),
    "n_test": len(TEST),
    "random_seeds": {
        "split_and_cv": SEED,
        "random_baseline": 0,
        "bootstrap_ci": 1,
        "paired_sign_flip_test": 2,
    },
    "budget_unit": "whitespace word-count proxy; not exact model tokens",
}
with open(os.path.join(CFG["out_dir"], "run_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2, default=str)
print("\nsaved to", CFG["out_dir"], ":", sorted(os.listdir(CFG["out_dir"])))

## 12. Limitations to state openly (in the paper, README and interviews)

1. **Retention is not LLM answer accuracy.** The downstream proxy is retrieval-based and string-matching. A true end-to-end test needs a reader model; run one on the held-out test questions before claiming QA gains.
2. **One synthetic pipeline.** LongMemEval-S evidence sessions come from a single generator, and the same-pipeline pool is a *stress test*, not a realistic memory. LoCoMo has only 10 conversations, so its power is low.
3. **Label quality.** Abstention questions are dropped, but other evidence labels may be imperfect; a corrected-label release of LongMemEval-S exists and should be swapped in and compared.
4. **Same-distribution test split.** The locked split removes selection bias and content overlap, but dev and test come from the same 500-question benchmark. It is not independent data.
5. **Unsupervised IDF was fit on all sessions** (no labels used), so test-distribution text contributes to vocabulary weighting. Refitting it on development-only sessions is a robustness check.
7. **The held-out split is not pristine.** The dev-based feature-group selection (section 5) was added after an earlier development run showed that the `time` and `style` groups hurt on this same test half. Treat all results as post-hoc held-out evidence and replicate on fresh data or a genuinely untouched split before making confirmatory claims.
6. **Hyper-parameters were tuned on dev only,** but pool definitions, features and the equivalence margin were chosen after earlier versions. Treat those as researcher degrees of freedom and say so.
7. **Question-level inference does not solve shared-session dependence.** Histories share sessions and the project audit found that all histories are connected; the question bootstrap/sign-flip test therefore does not provide independent-cluster uncertainty. Report this limitation and seek a benchmark with independent conversation clusters.

**If you want a stronger result than the audit itself:** add a harder, independent benchmark (for example one with semantically adjacent distractors), and run a reader LLM on the held-out test split.